# Arbeiten mit Klimadaten I: die Säkularstation in Potsdam

![alt text](../../img/saekularstation.png "Säkularstation Potsdam")

## Eure erste Amtshandlung in Python

In [ ]:
# Hallo Python-Welt
print("Hallo Python. Goodbye Excel.")

## Lade benötigte Pakete

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

*Recherchiere:* Wozu ist das Paket `pandas` gut? Wozu die Pakete `matplotlib` und `numpy`?

## Lade den Datensatz: über 130 Jahre Potsdamer Klimageschichte

Bevor Ihr den Code ausführt: Navigiert auf der linken Seite einmal in das Verzeichnis `../../data` und schaut Euch die Datei `klima-potsdam.txt` an. Das ist die Datengrundlage, mit der wir heute arbeiten.

In [ ]:
# Datei einlesen
#    verfügbar: klima-potsdam-03987.txt, klima-brocken-00722.txt,
#               klima-waghaeusel-kirrlach-05275, klima-helgoland-02115 
df = pd.read_csv("daten/klima-potsdam-03987.txt",# ../../ heißt zwei Ebenen über dem Arbeitsverzeichnis
                 sep=";",                        # Spaltentrenner
                 skipinitialspace=True,          # whitespace aus den Spaltennamen entfernen
                 na_values="-999")               # "-999" sind Fehlwerte
# Spalte "eor" rausschmeißen
df = df.drop(columns=["eor"])
# Spalte MESS_DATUM als Datum interpretieren
df["MESS_DATUM"] = pd.to_datetime(df.MESS_DATUM, format="%Y%m%d")
# MESS_DATUM als Zeilenindex verwenden
df = df.set_index("MESS_DATUM")

In [ ]:
# Und so sieht unser Datensatz aus (head and tail) - der Datentyp heißt "dataframe"
df

## Was bedeuten die Spalten?

Diese Info erhalten wir aus den [Metadaten](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/BESCHREIBUNG_obsgermany-climate-daily-kl_de.pdf), die vom DWD bereitgestellt werden.

- `FM`: Tagesmittel der Windgeschwindigkeit, m/s
- `FX`: Maximum der Windspitze Messnetz 3, m/sec
- `NM`: Tagesmittel des Bedeckungsgrades, Achtel
- `PM`: Tagesmittel des Luftdrucks, hpa
- `RSK`: tgl. Niederschlagshoehe, mm
- `RSKF`: tgl. Niederschlagsform
- `SDK`: Sonnenscheindauer Tagessumme, Stunden
- `SHK_TAG`: Schneehoehe Tageswert, cm
- `TGK`: Minimum der Lufttemperatur am Erdboden in 5cm, Grad C
- `TMK`: Tagesmittel der Temperatur, Grad C
- `TNK`: Tagesminimum der Lufttemperatur in 2m Hoehe, Grad C
- `TXK`: Tagesmaximum der Lufttemperatur in 2m Hoehe, Grad C
- `UPM`: Tagesmittel der Relativen Feuchte, Prozent
- `VPM`: Tagesmittel des Dampfdruckes, hpa

### Einfach mal das Tagesmittel der Lufttemperatur plotten

In [ ]:
df.TMK.plot()

Hm...das ist noch nicht besonders aussagekräftig.

## Beobachteter Klimawandel in Potsdam

Oder, genauer gesagt, "Klimaerwärmung in Potsdam" (Klimawandel ist nicht nur Erwärmung).

In [ ]:
# Zeitreihe der Jahresmitteltemperatur
dfamean = df.resample("YE").mean()
dfasum = df.resample("YE").sum()

In [ ]:
dfamean.TMK.plot()

Was sehen wir? Warum ist 2026 so kalt?

### Mögliche Aufgaben

Versuche nun mit Bastelei, Recherche und vor allem mit Hilfe von LLM (z.B. https://gptup.uni-potsdam.de/) folgende Aufgaben zu lösen.

1. Berechne den Mittelwert der Lufttemperatur in Potsdam für die gesamte Messreihe.
2. Berechne den Mittelwert der Lufttemperatur in Potsdam für die Klimanormalperioden 1931-1960, 1961-1990 und 1991-2020.
3. Verschönere die obige Abbildung: Füge eine y-Achsen-Beschriftung hinzu, entferne die x-Achsenbeschriftung, füge ein Gitter (`grid`) hinzu sowie horizontale Linien für die Werte in den Klimanormalperdioden.
4. Erstelle ähnliche Abbildungen für die mittlere Windgeschwindigkeit und die Jahresniederschlagssummme. Interpretation der Ergebnisse?
5. Wie hat sich die Zahl der Hitzetage (Tageshöchsttemperatur größer als 30 °C) pro Jahr entwickelt? Wieviel Hitzetage gab es im Mittel pro Jahr in den Klimanormalperioden 1961-1990 und 1991-2020? Interpretation? Wie viele Hitzetage gab es bislang im Jahr 2026?
6. Wie hat sich die Zahl der Tropennächte (Minimumtemperatur größer als 20 °C) pro Jahr entwickelt? Wieviel Tropennächte gab es im Mittel pro Jahr in den Klimanormalperioden 1961-1990 und 1991-2020? Interpretation? Wie viele Tropennächte gab es bislang im Jahr 2026?
7. Und nochmal das gleiche für Schneetage (Tage mit einer Schneehöhe von über 10 cm). Interpretation?
8. Was ist die höchste jemals gemessene Lufttemperatur in Potsdam? Wann wurde sie gemessen?
9. Erstelle eine Abbildung des mittleren Jahresgangs der Lufttemperatur für die Klimanormalperioden 1961-1990 und 1991-2020.

#### Nur ein paar Hinweise...

- Auf einen Zeitraum und eine Spalte greifst Du z.B. wie folgt zu: `df.loc["1961-01-01":"1990-01-01"]`
- Eine Summe oder einen Mittelwert erhaltst Du, indem Du z.B. `.sum()` oder `.mean()` hinter den DataFrame hängst.

In [ ]:
# Los geht's ...

## Das klassische Klimadiagramm nach Walter-Lieth

In [ ]:
startyear = "1991"
endyear = "2020"

In [ ]:
# Monatliche Werte (Mittelwert für Temperatur, Summe für Niederschlag)
monthly = df.loc[startyear:endyear].resample("ME").agg({
    "TMK": "mean",
    "RSK": "sum"
})

# Klimamittel über alle Jahre
climate = monthly.groupby(monthly.index.month).mean()
climate.columns = "T", "P"
climate.index.name = "Monat"
climate

In [ ]:
# Monatsnamen
months = ["Jan", "Feb", "Mär", "Apr", "Mai", "Jun",
          "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"]

# Plot
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.bar(climate.index, climate["P"], width=0.7, color="lightblue",
        label="Niederschlag")
ax1.set_ylabel("Niederschlag [mm]")
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(months)
#ax1.set_ylim(0, 200)

ax2 = ax1.twinx()
ax2.plot(climate.index, climate["T"], "o-", color="black",
         linewidth=2, label="Temperatur")
ax2.set_ylabel("Temperatur [°C]")
#ax2.set_ylim(-5, 25)

Tmean = climate["T"].mean()
Pmean = climate["P"].sum()
text = "Potsdam, %s-%s\n" % (startyear, endyear) + \
       "Temperatur: %.1f °C\n" % Tmean + \
       "Niederschlag: %.1f mm" % Pmean
    

ax1.text(
    0.98, 0.98,
    text,
    transform=ax1.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="none",
        alpha=0.8
    )
)

### Aufgabe

Fertige ein entsprechendes Klimadiagramm für den Zeitraum 1991-2020 für die Stationen auf Helgoland, auf dem Brocken und in Waghäusel-Kirrlach an (siehe Dateien im Verzeichnis `daten`). Mache jeweils einen Screenshot und stelle die Diagramme auf einer Folie zusammen. Was für Unterschiede fallen auf? Und wo ist Waghäusel-Kirrlach???  

## Wo gibt es weitere Klimastationen?

Der oben analysierte Datensatz stammt aus den [Open Data Repository des Deutschen Wetterdienstes](https://opendata.dwd.de/climate_environment/CDC/) (DWD). Ein unfassbarer Datenschatz, frei verfügbar für alle. Alles, was Ihr für die Station Potsdam oben angestellt habt, könnte Ihr ohne viel Aufhebens für alle anderen Stationen machen (natürlich sind die Messreihen nirgendwo so lang wie in Potsdam). 

Wir wollen nun darstellen, wo es weitere tägliche Klimastationsdaten gibt. Dafür schauen wir uns [diese Datei](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt) an.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import folium

In [ ]:
quelle = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt"
stationen = pd.read_fwf(quelle, encoding="ISO-8859-1", skiprows=2,
                        names=["id", "from", "to", "elevation", "lat", "lon", "name","state","abgabe"])
stationen

In [ ]:
gdf = gpd.GeoDataFrame(stationen, 
                       geometry=gpd.points_from_xy(stationen.lon, stationen.lat), crs="EPSG:4326")

In [ ]:
m = folium.Map(location=[51.4, 11.5], zoom_start=6, tiles="OpenStreetMap")

for _, row in stationen.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,#row["count"] / 2,
        color="black",#species_colours.get(row["species"], "grey"),
        fill=True,
        fill_opacity=0.5,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"Elevation: {row['elevation']} m<br>"
            f"ID: {row['id']}<br>",
            max_width=200,
        ),
    ).add_to(m)

m

#### Aufgabe

Versuche anhand der obigen Tabelle das Start- und Enddatum für jede Station zu ermitteln und in die angezeigten Attribute auf der Karte einzubeziehen. Vielleicht kannst Du sogar die Länge der Zeitreihen auf der Karte farbkodieren (anspruchsvoll!).